In [1]:
# %%
import sys, os
from dataclasses import dataclass, field
from typing import Any, Dict

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

print("PYTHONPATH:", PROJECT_ROOT)

from app.schemas import (
    PiiSanitizedBlock,
    RagResult,
    TailoredResume,
    VerificationResult,
    PdfArtifact
)

from app.pii.presidio_client import sanitize_text , LinkedInRecognizer
from app.rag.retriever import retrieve_ats_constraints
from app.pdf.generator import generate_pdf
from app.llm.optimizer import optimize_resume
from app.llm.verifier import verify_resume


PYTHONPATH: c:\Users\gyash\Desktop\virasat\Whatsapp_agent


c:\Users\gyash\Desktop\virasat\Whatsapp_agent\agent_wrapper\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4225.07it/s]


In [2]:
# %%
@dataclass
class FakeState:
    user_phone: str
    resume_text: str
    job_description: str
    sanitized_resume: str = ""
    sanitized_jd: str = ""
    rag_constraints: Any = None
    optimized_resume: str = ""
    verified_resume: str = ""
    pdf_path: str = ""

# In-memory state store
FAKE_DB = {}

def update_state(user_phone: str, **kwargs):
    if user_phone not in FAKE_DB:
        FAKE_DB[user_phone] = {}
    FAKE_DB[user_phone].update(kwargs)
    print(f"[STATE UPDATED] {kwargs}")


In [3]:
# %%
def pii_node(state):
    sanitized_resume = sanitize_text(state.resume_text)
    sanitized_jd = sanitize_text(state.job_description)

    update_state(
        state.user_phone,
        sanitized_resume=sanitized_resume.sanitized_text,
        sanitized_jd=sanitized_jd.sanitized_text
    )

    state.sanitized_resume = sanitized_resume.sanitized_text
    state.sanitized_jd = sanitized_jd.sanitized_text

    return {"sanitized_resume": state.sanitized_resume,
            "sanitized_jd": state.sanitized_jd}


def rag_node(state):
    result: RagResult = retrieve_ats_constraints(state.sanitized_jd)

    update_state(
        state.user_phone,
        rag_constraints=result.rules
    )

    state.rag_constraints = result.rules
    return {"rag_constraints": result.rules}


def optimize_node(state):
    optimized: TailoredResume = optimize_resume(
        resume=state.sanitized_resume,
        job_description=state.sanitized_jd,
        constraints=state.rag_constraints
    )

    update_state(
        state.user_phone,
        optimized_resume=optimized.optimized_text
    )

    state.optimized_resume = optimized.optimized_text
    return {"optimized_resume": optimized.optimized_text}


def verify_node(state):
    verified: VerificationResult = verify_resume(
        original=state.sanitized_resume,
        optimized=state.optimized_resume
    )

    update_state(
        state.user_phone,
        verified_resume=verified.cleaned_text
    )

    state.verified_resume = verified.cleaned_text
    return {"verified_resume": verified.cleaned_text}


def pdf_node(state):
    pdf: PdfArtifact = generate_pdf(
        user_phone=state.user_phone,
        text=state.verified_resume
    )

    update_state(
        state.user_phone,
        pdf_path=pdf.file_path
    )

    state.pdf_path = pdf.file_path
    return {"pdf_path": pdf.file_path}


In [4]:
# %%
state = FakeState(
    user_phone="12345",
    resume_text="""
     Yash Gandhi
832-652-7201 | ygash557@gmail.com | linkedin.com/in/yash-gandhi99 | Portfolio Page

TECHNICAL SKILLS
Programming Languages: Java, Python, C/C++, JavaScript, C#
Operating Systems: Red Hat Enterprise Linux, Windows, macOS
Web Development: ASP.NET Web API, React, Node.js, Flask, Tailwind
Database Environment: MySQL, NoSQL, Oracle, Postgres
Tools: Git, Docker, AWS, Google Cloud Platform, VS Code, Visual Studio, Jupyter Notebook, Google Colab
Libraries: pandas, NumPy, Matplotlib, Three-js
Certifications: Data Visualization with Python, Python for Data Science

EDUCATION
University of Houston Clear-Lake — Houston, TX
Bachelor of Science in Computer Science
Jan 2023 – Dec 2026
- Related Coursework: Network Programming, Mobile App Development, Web Development, Data Structures & Algorithm Analysis, Generative AI, Introduction to AI/ML, Big Data

PROJECTS
Virasat Technology | React, Tailwind, MongoDB, React-Three, API
March 2024 – June 2024
- Built an immersive, high-end interface using React and React-Three-Fiber for 3D visuals
- Improved performance using React.memo, lazy loading, and bundle optimization
- Created reusable hooks and utilities to standardize API calls and state management
- Processed and mapped MongoDB JSON data into structured UI components

CyberSwift | React, ThreeJs, Framer-motion, Auth0, MongoDB, Node.js, Socket.IO
Sep 2024 – Sep 2024
- Developed a 2D RPG-style learning platform with interactive visuals using React + Three.js
- Implemented real-time multiplayer features with Socket.IO
- Integrated Auth0 for secure authentication and user management
- Built backend services in Node.js to support game logic, user data, and progress tracking

EXPERIENCE
Full Stack Engineer
Self-Employed — Houston, TX
Jan 2025 – Present
- Designed and built scalable web apps with .NET and C#, focusing on strong server-side logic and modular component design
- Implemented Entity Framework to manage complex data schemas and ensure high-performance interactions with SQL-based backends
- Performed unit/system testing, debugging, and documentation to support stable releases

Full Stack Engineer
Alter Learning Educational Platform INC — Remote
Jan 2023 – June 2024
- Created React front-end with Python Flask backend services through custom middleware for consistent data flow

    """,
    job_description="""
    Looking for a Backend Engineer with Python, FastAPI, Docker, AWS,
    and experience with LLM pipelines.
    """
)


In [5]:
# %%
print("=== PII NODE ===")
pii_output = pii_node(state)
pii_output


=== PII NODE ===
[STATE UPDATED] {'sanitized_resume': '\n     Yash Gandhi\n<PII> | <PII> | <PII> | Portfolio Page\n\nTECHNICAL SKILLS\nProgramming Languages: Java, Python, C/C++, JavaScript, C#\nOperating Systems: Red Hat Enterprise Linux, Windows, macOS\nWeb Development: <PII> Web API, React, Node.js, Flask, Tailwind\nDatabase Environment: MySQL, NoSQL, Oracle, Postgres\nTools: Git, Docker, AWS, Google Cloud Platform, VS Code, Visual Studio, Jupyter Notebook, Google Colab\nLibraries: pandas, NumPy, Matplotlib, Three-js\nCertifications: Data Visualization with Python, Python for Data Science\n\nEDUCATION\nUniversity of Houston Clear-Lake — Houston, TX\nBachelor of Science in Computer Science\nJan 2023 – Dec 2026\n- Related Coursework: Network Programming, Mobile App Development, Web Development, Data Structures & Algorithm Analysis, Generative AI, Introduction to AI/ML, Big Data\n\nPROJECTS\nVirasat Technology | React, Tailwind, MongoDB, React-Three, API\nMarch 2024 – June 2024\n- Buil

{'sanitized_resume': '\n     Yash Gandhi\n<PII> | <PII> | <PII> | Portfolio Page\n\nTECHNICAL SKILLS\nProgramming Languages: Java, Python, C/C++, JavaScript, C#\nOperating Systems: Red Hat Enterprise Linux, Windows, macOS\nWeb Development: <PII> Web API, React, Node.js, Flask, Tailwind\nDatabase Environment: MySQL, NoSQL, Oracle, Postgres\nTools: Git, Docker, AWS, Google Cloud Platform, VS Code, Visual Studio, Jupyter Notebook, Google Colab\nLibraries: pandas, NumPy, Matplotlib, Three-js\nCertifications: Data Visualization with Python, Python for Data Science\n\nEDUCATION\nUniversity of Houston Clear-Lake — Houston, TX\nBachelor of Science in Computer Science\nJan 2023 – Dec 2026\n- Related Coursework: Network Programming, Mobile App Development, Web Development, Data Structures & Algorithm Analysis, Generative AI, Introduction to AI/ML, Big Data\n\nPROJECTS\nVirasat Technology | React, Tailwind, MongoDB, React-Three, API\nMarch 2024 – June 2024\n- Built an immersive, high-end interfac

In [6]:
# %%
print("=== RAG NODE ===")
rag_output = rag_node(state)
rag_output


=== RAG NODE ===
[STATE UPDATED] {'rag_constraints': ['Job titles should be written in a standard format so ATS can correctly parse your experience.', 'Avoid keyword stuffing in your resume. Overusing the same keyword can be penalized by ATS.', 'Keep resume length to one or two pages for optimal ATS scanning.', 'Use action verbs at the start of each bullet to improve ATS parsing.', 'ATS systems may not parse images or graphics. Avoid using icons, logos, or decorative elements.']}


{'rag_constraints': ['Job titles should be written in a standard format so ATS can correctly parse your experience.',
  'Avoid keyword stuffing in your resume. Overusing the same keyword can be penalized by ATS.',
  'Keep resume length to one or two pages for optimal ATS scanning.',
  'Use action verbs at the start of each bullet to improve ATS parsing.',
  'ATS systems may not parse images or graphics. Avoid using icons, logos, or decorative elements.']}

In [7]:
# %%
print("=== OPTIMIZE NODE ===")
opt_output = optimize_node(state)
opt_output


=== OPTIMIZE NODE ===
[STATE UPDATED] {'optimized_resume': 'Yash Gandhi\n<PII> | <PII> | <PII> | Portfolio Page\n\nTECHNICAL SKILLS\nProgramming Languages: Python, Java, C/C++, JavaScript, C#  \nOperating Systems: Red Hat Enterprise Linux, Windows, macOS  \nWeb Development: FastAPI, Flask, React, Node.js  \nDatabase Environment: MySQL, NoSQL, Oracle, Postgres  \nTools: Docker, AWS, Git, Google Cloud Platform, VS Code, Visual Studio  \nLibraries: pandas, NumPy, Matplotlib  \nCertifications: Data Visualization with Python, Python for Data Science\n\nEDUCATION\nUniversity of Houston Clear-Lake — Houston, TX  \nBachelor of Science in Computer Science  \nJan 2023 – Dec 2026  \n- Related Coursework: Network Programming, Mobile App Development, Web Development, Data Structures & Algorithm Analysis, Generative AI, Introduction to AI/ML, Big Data\n\nPROJECTS\nVirasat Technology | React, Tailwind, MongoDB, React-Three, API  \nMarch 2024 – June 2024  \n- Developed an immersive interface using Rea

{'optimized_resume': 'Yash Gandhi\n<PII> | <PII> | <PII> | Portfolio Page\n\nTECHNICAL SKILLS\nProgramming Languages: Python, Java, C/C++, JavaScript, C#  \nOperating Systems: Red Hat Enterprise Linux, Windows, macOS  \nWeb Development: FastAPI, Flask, React, Node.js  \nDatabase Environment: MySQL, NoSQL, Oracle, Postgres  \nTools: Docker, AWS, Git, Google Cloud Platform, VS Code, Visual Studio  \nLibraries: pandas, NumPy, Matplotlib  \nCertifications: Data Visualization with Python, Python for Data Science\n\nEDUCATION\nUniversity of Houston Clear-Lake — Houston, TX  \nBachelor of Science in Computer Science  \nJan 2023 – Dec 2026  \n- Related Coursework: Network Programming, Mobile App Development, Web Development, Data Structures & Algorithm Analysis, Generative AI, Introduction to AI/ML, Big Data\n\nPROJECTS\nVirasat Technology | React, Tailwind, MongoDB, React-Three, API  \nMarch 2024 – June 2024  \n- Developed an immersive interface using React and React-Three-Fiber for 3D visual

In [8]:
# %%
print("=== VERIFY NODE ===")
verify_output = verify_node(state)
verify_output


=== VERIFY NODE ===
[STATE UPDATED] {'verified_resume': 'Yash Gandhi  \n<PII> | <PII> | <PII> | Portfolio Page  \n\nTECHNICAL SKILLS  \nProgramming Languages: Java, Python, C/C++, JavaScript, C#  \nOperating Systems: Red Hat Enterprise Linux, Windows, macOS  \nWeb Development: Flask, React, Node.js  \nDatabase Environment: MySQL, NoSQL, Oracle, Postgres  \nTools: Git, Docker, AWS, Google Cloud Platform, VS Code, Visual Studio  \nLibraries: pandas, NumPy, Matplotlib, Three-js  \nCertifications: Data Visualization with Python, Python for Data Science  \n\nEDUCATION  \nUniversity of Houston Clear-Lake — Houston, TX  \nBachelor of Science in Computer Science  \nJan 2023 – Dec 2026  \n- Related Coursework: Network Programming, Mobile App Development, Web Development, Data Structures & Algorithm Analysis, Generative AI, Introduction to AI/ML, Big Data  \n\nPROJECTS  \nVirasat Technology | React, Tailwind, MongoDB, React-Three, API  \nMarch 2024 – June 2024  \n- Built an immersive, high-end i

{'verified_resume': 'Yash Gandhi  \n<PII> | <PII> | <PII> | Portfolio Page  \n\nTECHNICAL SKILLS  \nProgramming Languages: Java, Python, C/C++, JavaScript, C#  \nOperating Systems: Red Hat Enterprise Linux, Windows, macOS  \nWeb Development: Flask, React, Node.js  \nDatabase Environment: MySQL, NoSQL, Oracle, Postgres  \nTools: Git, Docker, AWS, Google Cloud Platform, VS Code, Visual Studio  \nLibraries: pandas, NumPy, Matplotlib, Three-js  \nCertifications: Data Visualization with Python, Python for Data Science  \n\nEDUCATION  \nUniversity of Houston Clear-Lake — Houston, TX  \nBachelor of Science in Computer Science  \nJan 2023 – Dec 2026  \n- Related Coursework: Network Programming, Mobile App Development, Web Development, Data Structures & Algorithm Analysis, Generative AI, Introduction to AI/ML, Big Data  \n\nPROJECTS  \nVirasat Technology | React, Tailwind, MongoDB, React-Three, API  \nMarch 2024 – June 2024  \n- Built an immersive, high-end interface using React and React-Three

In [9]:
# %%
print("=== PDF NODE ===")
pdf_output = pdf_node(state)
pdf_output


=== PDF NODE ===
[STATE UPDATED] {'pdf_path': 'data/output/12345/optimized_resume.pdf'}


{'pdf_path': 'data/output/12345/optimized_resume.pdf'}

In [10]:
# %%
print("=== FULL PIPELINE ===")

state = FakeState(
    user_phone="99999",
    resume_text="""
Yash Gandhi
832-652-7201 | ygash557@gmail.com | linkedin.com/in/yash-gandhi99 | Portfolio Page

TECHNICAL SKILLS
Programming Languages: Java, Python, C/C++, JavaScript, C#
Operating Systems: Red Hat Enterprise Linux, Windows, macOS
Web Development: ASP.NET Web API, React, Node.js, Flask, Tailwind
Database Environment: MySQL, NoSQL, Oracle, Postgres
Tools: Git, Docker, AWS, Google Cloud Platform, VS Code, Visual Studio, Jupyter Notebook, Google Colab
Libraries: pandas, NumPy, Matplotlib, Three-js
Certifications: Data Visualization with Python, Python for Data Science

EDUCATION
University of Houston Clear-Lake — Houston, TX
Bachelor of Science in Computer Science
Jan 2023 – Dec 2026
- Related Coursework: Network Programming, Mobile App Development, Web Development, Data Structures & Algorithm Analysis, Generative AI, Introduction to AI/ML, Big Data

PROJECTS
Virasat Technology | React, Tailwind, MongoDB, React-Three, API
March 2024 – June 2024
- Built an immersive, high-end interface using React and React-Three-Fiber for 3D visuals
- Improved performance using React.memo, lazy loading, and bundle optimization
- Created reusable hooks and utilities to standardize API calls and state management
- Processed and mapped MongoDB JSON data into structured UI components

CyberSwift | React, ThreeJs, Framer-motion, Auth0, MongoDB, Node.js, Socket.IO
Sep 2024 – Sep 2024
- Developed a 2D RPG-style learning platform with interactive visuals using React + Three.js
- Implemented real-time multiplayer features with Socket.IO
- Integrated Auth0 for secure authentication and user management
- Built backend services in Node.js to support game logic, user data, and progress tracking

EXPERIENCE
Full Stack Engineer
Self-Employed — Houston, TX
Jan 2025 – Present
- Designed and built scalable web apps with .NET and C#, focusing on strong server-side logic and modular component design
- Implemented Entity Framework to manage complex data schemas and ensure high-performance interactions with SQL-based backends
- Performed unit/system testing, debugging, and documentation to support stable releases

Full Stack Engineer
Alter Learning Educational Platform INC — Remote
Jan 2023 – June 2024
- Created React front-end with Python Flask backend services through custom middleware for consistent data flow
""",
    job_description="Looking for a backend engineer with Python, FastAPI, AWS, and LLM experience."
)

pii_node(state)
rag_node(state)
optimize_node(state)
verify_node(state)
pdf_node(state)

print("\nFinal State:")
FAKE_DB["99999"]


=== FULL PIPELINE ===
[STATE UPDATED] {'sanitized_resume': '\nYash Gandhi\n<PII> | <PII> | <PII> | Portfolio Page\n\nTECHNICAL SKILLS\nProgramming Languages: Java, Python, C/C++, JavaScript, C#\nOperating Systems: Red Hat Enterprise Linux, Windows, macOS\nWeb Development: <PII> Web API, React, Node.js, Flask, Tailwind\nDatabase Environment: MySQL, NoSQL, Oracle, Postgres\nTools: Git, Docker, AWS, Google Cloud Platform, VS Code, Visual Studio, Jupyter Notebook, Google Colab\nLibraries: pandas, NumPy, Matplotlib, Three-js\nCertifications: Data Visualization with Python, Python for Data Science\n\nEDUCATION\nUniversity of Houston Clear-Lake — Houston, TX\nBachelor of Science in Computer Science\nJan 2023 – Dec 2026\n- Related Coursework: Network Programming, Mobile App Development, Web Development, Data Structures & Algorithm Analysis, Generative AI, Introduction to AI/ML, Big Data\n\nPROJECTS\nVirasat Technology | React, Tailwind, MongoDB, React-Three, API\nMarch 2024 – June 2024\n- Buil

{'sanitized_resume': '\nYash Gandhi\n<PII> | <PII> | <PII> | Portfolio Page\n\nTECHNICAL SKILLS\nProgramming Languages: Java, Python, C/C++, JavaScript, C#\nOperating Systems: Red Hat Enterprise Linux, Windows, macOS\nWeb Development: <PII> Web API, React, Node.js, Flask, Tailwind\nDatabase Environment: MySQL, NoSQL, Oracle, Postgres\nTools: Git, Docker, AWS, Google Cloud Platform, VS Code, Visual Studio, Jupyter Notebook, Google Colab\nLibraries: pandas, NumPy, Matplotlib, Three-js\nCertifications: Data Visualization with Python, Python for Data Science\n\nEDUCATION\nUniversity of Houston Clear-Lake — Houston, TX\nBachelor of Science in Computer Science\nJan 2023 – Dec 2026\n- Related Coursework: Network Programming, Mobile App Development, Web Development, Data Structures & Algorithm Analysis, Generative AI, Introduction to AI/ML, Big Data\n\nPROJECTS\nVirasat Technology | React, Tailwind, MongoDB, React-Three, API\nMarch 2024 – June 2024\n- Built an immersive, high-end interface usi